# Trading App v2: HITS Hub/Authority vs DP Oracle

This is the end-to-end experiment requested for the $1T, $100B, and $10B universes. Every tier uses the same code path: build 2024 graph/DP labels, train classifiers on 2024 features, score 2025, and run a next-open entry/exit backtest. HITS hub labels represent entry dates; HITS authority labels represent exit dates.

In [ ]:
from __future__ import annotations

from pathlib import Path
import json
import os
import sys
from time import perf_counter

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.ensemble import RandomForestClassifier

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / 'app').is_dir() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
WORKSPACE_ROOT = REPO_ROOT.parent
for package, repo in [('quant_warehouse', WORKSPACE_ROOT / 'quant-warehouse'), ('quant_orchestrator', WORKSPACE_ROOT / 'quant-orchestrator')]:
    if repo.exists() and str(repo) not in sys.path:
        sys.path.insert(0, str(repo))

from quant_warehouse.research_tools import FamilyEvaluationConfig, screen_fmp_equity_universe
from quant_warehouse.platforms.data_providers.fmp.target_engineering import LabelBuildSpec, build_trade_results
from quant_warehouse.warehouse.api import Warehouse

TRAIN_START = pd.Timestamp('2024-01-01')
TRAIN_END = pd.Timestamp('2024-12-31')
TEST_START = pd.Timestamp('2025-01-01')
TEST_END = pd.Timestamp('2025-12-31')
TIER_CONFIGS = {'1T': 1_000_000_000_000, '100B': 100_000_000_000, '10B': 10_000_000_000}
MIN_RETURN = 0.01
MAX_HOLDING_DAYS = 120
LABEL_QUANTILE = 0.80
BACKTEST_MAX_HOLD_DAYS = 20
BACKTEST_COST_BPS = 5.5
DP_K = 3
RF_ESTIMATORS = int(os.getenv('GRAPH_ORACLE_RF_ESTIMATORS', '120'))
RANDOM_SEED = 20260715
OUTPUT_DIR = REPO_ROOT / 'artifacts' / 'graph_oracle_wfo'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


## Label construction

For each calendar year and symbol, every `entry_date < exit_date` pair becomes an edge. Weighted HITS produces entry hubs and exit authorities separately for long and short returns. DP labels use the repository's existing non-overlapping strategy solver with `YE k=3`.

In [ ]:
FEATURE_COLS = ['ret_1', 'ret_5', 'ret_20', 'ret_60', 'vol_20', 'vol_60', 'dist_ma_20', 'dist_ma_60', 'range_20']
TARGETS = {
    'hits': {'long_entry': 'hits_long_hub_label', 'long_exit': 'hits_long_authority_label', 'short_entry': 'hits_short_hub_label', 'short_exit': 'hits_short_authority_label'},
    'dp': {'long_entry': 'dp_long_entry_label', 'long_exit': 'dp_long_exit_label', 'short_entry': 'dp_short_entry_label', 'short_exit': 'dp_short_exit_label'},
}

def _price_frame(raw: pd.DataFrame) -> pd.DataFrame:
    frame = raw.copy()
    if 'date' not in frame.columns:
        frame = frame.reset_index()
    frame.columns = [str(c).lower() for c in frame.columns]
    date_col = 'date' if 'date' in frame.columns else frame.columns[0]
    frame = frame.rename(columns={date_col: 'date'})
    frame['date'] = pd.to_datetime(frame['date'], errors='coerce').dt.normalize()
    needed = ['date', 'open', 'high', 'low', 'close']
    missing = [c for c in needed if c not in frame.columns]
    if missing:
        raise KeyError(f'Price frame missing {missing}')
    return frame[needed].dropna(subset=['date']).sort_values('date').drop_duplicates('date').reset_index(drop=True)

def _normalize_score(values: np.ndarray) -> np.ndarray:
    norm = np.linalg.norm(values)
    return values / (norm if norm > 0 else 1.0)

def _hits_labels(frame: pd.DataFrame) -> pd.DataFrame:
    work = frame.sort_values('date').reset_index(drop=True)
    n = len(work)
    dates = work['date'].to_numpy()
    high = pd.to_numeric(work['high'], errors='coerce').to_numpy(float)
    low = pd.to_numeric(work['low'], errors='coerce').to_numpy(float)
    forward = np.triu(np.ones((n, n), dtype=bool), k=1)
    horizon = np.arange(n)[None, :] - np.arange(n)[:, None]
    valid = forward & (horizon <= MAX_HOLDING_DAYS)
    long_returns = low[None, :] / high[:, None] - 1.0
    short_returns = low[:, None] / high[None, :] - 1.0
    parts = {'date': dates}
    for side, returns in [('long', long_returns), ('short', short_returns)]:
        weights = np.where(valid, np.maximum(returns - MIN_RETURN, 0.0), 0.0)
        hub = np.ones(n, dtype=float)
        authority = np.ones(n, dtype=float)
        for _ in range(40):
            authority = _normalize_score(weights.T @ hub)
            hub = _normalize_score(weights @ authority)
        hub_threshold = np.quantile(hub[hub > 0], LABEL_QUANTILE) if (hub > 0).any() else np.inf
        authority_threshold = np.quantile(authority[authority > 0], LABEL_QUANTILE) if (authority > 0).any() else np.inf
        parts[f'hits_{side}_hub_score'] = hub
        parts[f'hits_{side}_authority_score'] = authority
        parts[f'hits_{side}_hub_label'] = (hub >= hub_threshold).astype('uint8')
        parts[f'hits_{side}_authority_label'] = (authority >= authority_threshold).astype('uint8')
        parts[f'hits_{side}_positive_edge_count'] = (weights > 0).sum(axis=1)
    return pd.DataFrame(parts)

def _dp_labels(frame: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    dates = frame['date'].tolist()
    out = pd.DataFrame({'date': dates})
    for column in TARGETS['dp'].values():
        out[column] = 0
    spec = LabelBuildSpec(k_params={'YE': [DP_K]}, min_profit_pct=MIN_RETURN, buy_execution='high', sell_execution='low', short_execution='low', cover_execution='high')
    result = build_trade_results(['SYMBOL'], spec=spec, price_frames={'SYMBOL': frame})
    trades = pd.DataFrame(result.completed_trades)
    if trades.empty:
        return out, trades
    for side in ('long', 'short'):
        side_trades = trades.loc[trades['side'].astype(str).str.lower().eq(side)]
        entries = set(pd.to_datetime(side_trades['entry_date'], errors='coerce').dt.normalize())
        exits = set(pd.to_datetime(side_trades['exit_date'], errors='coerce').dt.normalize())
        out.loc[out['date'].isin(entries), f'dp_{side}_entry_label'] = 1
        out.loc[out['date'].isin(exits), f'dp_{side}_exit_label'] = 1
    return out, trades

def _features(frame: pd.DataFrame) -> pd.DataFrame:
    out = frame[['date', 'close', 'high', 'low']].copy()
    close = pd.to_numeric(out['close'], errors='coerce')
    out['ret_1'] = close.pct_change()
    out['ret_5'] = close.pct_change(5)
    out['ret_20'] = close.pct_change(20)
    out['ret_60'] = close.pct_change(60)
    out['vol_20'] = out['ret_1'].rolling(20).std()
    out['vol_60'] = out['ret_1'].rolling(60).std()
    out['dist_ma_20'] = close / close.rolling(20).mean() - 1.0
    out['dist_ma_60'] = close / close.rolling(60).mean() - 1.0
    out['range_20'] = (out['high'] / out['low'] - 1.0).rolling(20).mean()
    return out[['date', *FEATURE_COLS]].dropna()


## OOS backtest and tier runner

The classifier receives only 2024 rows. A signal on a 2025 date executes at the next day's open. Positions exit on the predicted authority label or after 20 trading days. The raw DP oracle is reported separately as a hindsight upper-bound reference.

In [ ]:
def _drawdown(returns: list[float]) -> float:
    if not returns:
        return np.nan
    curve = np.cumprod(1.0 + np.asarray(returns, dtype=float))
    peak = np.maximum.accumulate(curve)
    return float(np.min(curve / peak - 1.0))

def _backtest(test_rows: pd.DataFrame, prices: dict[str, pd.DataFrame], entry_col: str, exit_col: str, side: str) -> dict[str, object]:
    trades = []
    for symbol, group in test_rows.groupby('symbol', sort=True):
        price = prices[symbol].sort_values('date').reset_index(drop=True)
        score = group.set_index('date')[[entry_col, exit_col]].reindex(price['date']).fillna(0.0)
        active = None
        for i in range(len(price) - 1):
            if active is None and float(score.iloc[i][entry_col]) >= 0.5:
                active = i + 1
                continue
            if active is not None:
                holding = i - active + 1
                should_exit = float(score.iloc[i][exit_col]) >= 0.5 or holding >= BACKTEST_MAX_HOLD_DAYS
                if should_exit and i + 1 > active:
                    entry_px = float(price.iloc[active]['open'])
                    exit_px = float(price.iloc[i + 1]['open'])
                    raw_return = exit_px / entry_px - 1.0 if side == 'long' else entry_px / exit_px - 1.0
                    net_return = raw_return - 2.0 * BACKTEST_COST_BPS / 10_000.0
                    trades.append({'symbol': symbol, 'entry_date': price.iloc[active]['date'], 'exit_date': price.iloc[i + 1]['date'], 'return': net_return})
                    active = None
        if active is not None and active < len(price) - 1:
            entry_px = float(price.iloc[active]['open'])
            exit_px = float(price.iloc[-1]['close'])
            raw_return = exit_px / entry_px - 1.0 if side == 'long' else entry_px / exit_px - 1.0
            trades.append({'symbol': symbol, 'entry_date': price.iloc[active]['date'], 'exit_date': price.iloc[-1]['date'], 'return': raw_return - 2.0 * BACKTEST_COST_BPS / 10_000.0})
    frame = pd.DataFrame(trades)
    returns = frame['return'].astype(float).tolist() if not frame.empty else []
    total = float(np.prod(1.0 + np.asarray(returns)) - 1.0) if returns else np.nan
    return {'trades': len(returns), 'total_return': total, 'mean_return': float(np.mean(returns)) if returns else np.nan, 'win_rate': float(np.mean(np.asarray(returns) > 0)) if returns else np.nan, 'max_drawdown': _drawdown(returns), 'trade_frame': frame}

def _oracle_summary(trades: pd.DataFrame) -> dict[str, object]:
    if trades.empty:
        return {'trades': 0, 'total_return': np.nan, 'mean_return': np.nan, 'win_rate': np.nan, 'max_drawdown': np.nan}
    returns = []
    for _, row in trades.iterrows():
        entry = float(str(row['entry_px']).replace(',', ''))
        exit_ = float(str(row['exit_px']).replace(',', ''))
        returns.append(exit_ / entry - 1.0 if str(row['side']).lower() == 'long' else entry / exit_ - 1.0)
    return {'trades': len(returns), 'total_return': float(np.prod(1.0 + np.asarray(returns)) - 1.0), 'mean_return': float(np.mean(returns)), 'win_rate': float(np.mean(np.asarray(returns) > 0)), 'max_drawdown': _drawdown(returns)}

def run_tier(tier_name: str, min_market_cap: int) -> tuple[pd.DataFrame, pd.DataFrame]:
    started = perf_counter()
    warehouse = Warehouse()
    config = FamilyEvaluationConfig(market_cap_min=min_market_cap, start_date=str(TRAIN_START.date()), end_date=str(TEST_END.date()))
    universe, _, _, universe_source = screen_fmp_equity_universe(config, warehouse=warehouse, required_sections=('prices',))
    prices = {}
    for raw_symbol in universe:
        symbol = str(raw_symbol).upper()
        raw = warehouse.read_prices(symbol, provider='fmp', start=str(TRAIN_START.date()), end=str(TEST_END.date()))
        if raw is None or raw.empty:
            continue
        try:
            frame = _price_frame(raw)
        except KeyError:
            continue
        if len(frame) >= 100:
            prices[symbol] = frame
    symbols = sorted(prices)
    label_parts, feature_parts, dp_trade_parts = [], [], []
    for symbol in symbols:
        frame = prices[symbol]
        feature = _features(frame)
        feature_parts.append(feature.assign(symbol=symbol))
        for year, year_frame in frame.groupby(frame['date'].dt.year):
            if year not in (2024, 2025) or len(year_frame) < 30:
                continue
            hits = _hits_labels(year_frame)
            dp, dp_trades = _dp_labels(year_frame)
            labels = hits.merge(dp, on='date', how='left')
            labels.insert(0, 'symbol', symbol)
            labels['year'] = int(year)
            label_parts.append(labels)
            if not dp_trades.empty:
                dp_trades['symbol'] = symbol
                dp_trade_parts.append(dp_trades)
    labels = pd.concat(label_parts, ignore_index=True) if label_parts else pd.DataFrame()
    features = pd.concat(feature_parts, ignore_index=True) if feature_parts else pd.DataFrame()
    dp_trades = pd.concat(dp_trade_parts, ignore_index=True) if dp_trade_parts else pd.DataFrame()
    if labels.empty or features.empty:
        return pd.DataFrame([{'tier': tier_name, 'status': 'no_data', 'symbols': len(symbols)}]), pd.DataFrame()
    data = features.merge(labels, on=['symbol', 'date'], how='inner')
    train = data.loc[data['date'].between(TRAIN_START, TRAIN_END)].copy()
    test = data.loc[data['date'].between(TEST_START, TEST_END)].copy()
    result_rows, diagnostics = [], []
    for source, target_map in TARGETS.items():
        models = {}
        for role, target in target_map.items():
            if train[target].nunique() < 2:
                diagnostics.append({'tier': tier_name, 'source': source, 'role': role, 'status': 'insufficient_train_classes', 'positive_rate': float(train[target].mean())})
                continue
            model = RandomForestClassifier(n_estimators=RF_ESTIMATORS, max_depth=12, min_samples_leaf=5, class_weight='balanced_subsample', random_state=RANDOM_SEED, n_jobs=-1)
            model.fit(train[FEATURE_COLS], train[target])
            models[role] = model
        for side in ('long', 'short'):
            entry_role, exit_role = f'{side}_entry', f'{side}_exit'
            if entry_role not in models or exit_role not in models:
                continue
            scored = test[['symbol', 'date']].copy()
            scored[f'{source}_{side}_entry_score'] = models[entry_role].predict_proba(test[FEATURE_COLS])[:, 1]
            scored[f'{source}_{side}_exit_score'] = models[exit_role].predict_proba(test[FEATURE_COLS])[:, 1]
            bt = _backtest(scored, prices, f'{source}_{side}_entry_score', f'{source}_{side}_exit_score', side)
            result_rows.append({'tier': tier_name, 'source': source, 'side': side, 'status': 'ok', 'train_rows': len(train), 'test_rows': len(test), 'train_positive_rate': float(train[target_map[entry_role]].mean()), **{k: v for k, v in bt.items() if k != 'trade_frame'}})
    test_dp = dp_trades.loc[(pd.to_datetime(dp_trades['entry_date'], errors='coerce') >= TEST_START) & (pd.to_datetime(dp_trades['entry_date'], errors='coerce') <= TEST_END) & (pd.to_numeric(dp_trades['k'], errors='coerce') == DP_K)].copy() if not dp_trades.empty else pd.DataFrame()
    oracle = _oracle_summary(test_dp)
    result_rows.append({'tier': tier_name, 'source': 'raw_dp_oracle', 'side': 'both', 'status': 'oracle_reference', **oracle})
    diagnostics.append({'tier': tier_name, 'source': 'run', 'role': 'all', 'status': 'ok', 'symbols': len(symbols), 'universe_source': universe_source, 'elapsed_seconds': perf_counter() - started, 'train_rows': len(train), 'test_rows': len(test), 'dp_test_trades': len(test_dp)})
    return pd.DataFrame(result_rows), pd.DataFrame(diagnostics)

all_results, all_diagnostics = [], []
for tier_name, min_market_cap in TIER_CONFIGS.items():
    print(f'===== {tier_name} / ${min_market_cap:,} =====', flush=True)
    result, diagnostics = run_tier(tier_name, min_market_cap)
    all_results.append(result)
    all_diagnostics.append(diagnostics)
    display(result.drop(columns=['trade_frame'], errors='ignore'))
results = pd.concat(all_results, ignore_index=True)
diagnostics = pd.concat(all_diagnostics, ignore_index=True)
results.to_csv(OUTPUT_DIR / 'hits_vs_dp_results.csv', index=False)
diagnostics.to_csv(OUTPUT_DIR / 'hits_vs_dp_diagnostics.csv', index=False)
display(diagnostics)
display(results)

## Decision rule

HITS is better for a tier only if its OOS 2025 strategy has stronger total/mean return and acceptable drawdown than the DP-label strategy, while the raw DP oracle remains a hindsight reference. A classifier metric alone is not used to declare a winner.